<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [1]:
#%pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

In [2]:
# Params
VERBOSE = False
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged', '7_wonders']
IT = 5
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'
OVERWRITE = ['']
try:
    #DRIVE
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    BASE_FOLDER = 'drive/MyDrive/NLP_proj/explain/'
    N_GPU_LAYERS = -1
except:
    #LOCAL
    BASE_FOLDER = 'explain/'
    N_GPU_LAYERS = 20

In [3]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os

# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q6_K.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"bartowski/Qwen2.5-7B-Instruct-GGUF",
             'filename': "Qwen2.5-7B-Instruct-Q6_K.gguf",
             'temperature': 0.6,
             'n_ctx': 32768,
             'chat_format': "qwen"},
    'gemma': {'repo_id':"bartowski/google_gemma-3n-E4B-it-GGUF",
                'filename':"google_gemma-3n-E4B-it-Q6_K.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': None
             },
}
model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=N_GPU_LAYERS, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=VERBOSE,
                            force_download=True,
                            enable_thinking=True)

def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]

./Meta-Llama-3.1-8B-Instruct-Q6_K.gguf:   0%|          | 0.00/6.60G [00:00<?, ?B/s]

llama_context: n_ctx_per_seq (32768) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


# Summarize
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [4]:
ages = [7,11,16]
prompt = """You are a friendly tutor explaining board games to a {}‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
    - Goal of the game
    - How a player wins
    - What a turn looks like
    - Exceptions to standard rules

The user will give you a text file with the rulebook you need to explain.
the output should not be too long. All main rules must be present in your explanation."""

output_dict = {str(a): {str(it): '' for it in range(IT) } for a in ages}
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                                 or f'{CHOSEN}_{g}.json' in OVERWRITE]

if to_do == []:
    print('Nothing to do')
    
for g in to_do:
    for age,it in tqdm([(a,it) for a in ages for it in range(IT)]):
        rulebook = requests.get(BASE_URL+g+'.txt').text
        name = g.replace('_',' ')
        sys_prompt = prompt.format(age)
        usr_prompt = f'Here is the complete rulebook for the game {name}. Explain it to a {age}-year old. \n' + rulebook
        out = model.create_chat_completion(generate_message(sys_prompt, usr_prompt), 
                                           temperature=models[CHOSEN]['temperature'])['choices'][0]['message']['content']
        output_dict[str(age)][str(it)] = out
        if(VERBOSE):
            print(f'{g} age {age}, iteration: {it}\n---------\n{out}')
        

    with open(f'{BASE_FOLDER}{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

100%|███████████████████████████████████████████| 15/15 [24:57<00:00, 99.86s/it]
